# Лабораторная работа №3
## CNN для распознавания визуальных объектов образовательной среды

**Студенческий шаблон — PyTorch / Google Colab**

Цель: сравнить полносвязную сеть и сверточную нейронную сеть на индивидуальной подвыборке **EMNIST Balanced** и объяснить преимущество использования локальной пространственной структуры изображения.

Максимальная оценка: **20 баллов**.

## Теория 1. Свертка и карта признаков

Для двумерного входа сверточный слой вычисляет локальную взвешенную сумму:

$$
Y_{c,i,j}=b_c+\sum_d\sum_u\sum_v K_{c,d,u,v}X_{d,i+u,j+v}
$$

Один и тот же фильтр используется во всех позициях изображения. Это свойство называется **разделением весов**.

Число параметров:

$$
N_{conv}=(K_hK_wC_{in}+1)C_{out}
$$

Свертка обладает **эквивариантностью к сдвигу**: при переносе объекта карта признаков также переносится. Пулинг и аугментация делают итоговую классификацию приближенно устойчивой к небольшим сдвигам.

## Теория 2. Пулинг и размеры тензоров

`MaxPool2d(2)` оставляет максимум в каждом окне $2\times2$.

Для изображения $28\times28$:

```text
28×28 -> Conv -> 28×28 -> Pool -> 14×14
14×14 -> Conv -> 14×14 -> Pool -> 7×7
```

Размер выхода свертки:

$$
H_{out}=\left\lfloor\frac{H_{in}+2P-D(K-1)-1}{S}+1\right\rfloor
$$

В нашей архитектуре `kernel_size=3`, `padding=1`, `stride=1`, поэтому свертки сохраняют пространственный размер.

## Теория 3. CrossEntropyLoss и обратное распространение

PyTorch `nn.CrossEntropyLoss()` получает **логиты**, а не вероятности:

$$
L_i=-\log\frac{\exp(z_{i,y_i})}{\sum_c\exp(z_{i,c})}
$$

Поэтому **не добавляйте Softmax в `forward()`**.

Обновление параметров:

$$
\theta_{t+1}=\theta_t-\eta\nabla_\theta L
$$

В PyTorch:

```python
loss.backward()     # вычислить градиенты
optimizer.step()    # обновить параметры
```

В работе используется Adam.

## Шаг 1. Импорт библиотек и проверка среды

Запустите ячейку. Для Colab рекомендуется выбрать **Runtime → Change runtime type → GPU**.

In [ ]:
# Основные библиотеки
import random
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Требование лабораторной: PyTorch 2.0+
major = int(torch.__version__.split(".")[0])
assert major >= 2, "Требуется PyTorch 2.0 или новее."

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:
def set_seed(seed: int = 42):
    """Фиксируем основные источники случайности."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Для повторяемости на одной и той же платформе.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def seed_worker(worker_id):
    """Синхронизация seed у процессов DataLoader."""
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Шаг 2. Выбор варианта

Измените только `V`. Все остальные параметры должны быть получены из таблицы вариантов.

In [ ]:
# Единая таблица параметров 25 вариантов.
VARIANTS = {
  "1": {
    "classes": [
      "0",
      "1",
      "2",
      "3",
      "4",
      "5",
      "6",
      "7",
      "8",
      "9"
    ],
    "filters1": 16,
    "rotation": 5,
    "dropout": 0.1,
    "budget": 24000
  },
  "2": {
    "classes": [
      "A",
      "B",
      "C",
      "D",
      "E",
      "F",
      "G",
      "H",
      "I",
      "J"
    ],
    "filters1": 24,
    "rotation": 10,
    "dropout": 0.2,
    "budget": 40000
  },
  "3": {
    "classes": [
      "K",
      "L",
      "M",
      "N",
      "O",
      "P",
      "Q",
      "R",
      "S",
      "T"
    ],
    "filters1": 32,
    "rotation": 15,
    "dropout": 0.3,
    "budget": 60000
  },
  "4": {
    "classes": [
      "Q",
      "R",
      "S",
      "T",
      "U",
      "V",
      "W",
      "X",
      "Y",
      "Z"
    ],
    "filters1": 8,
    "rotation": 20,
    "dropout": 0.4,
    "budget": 12000
  },
  "5": {
    "classes": [
      "a",
      "b",
      "d",
      "e",
      "f",
      "g",
      "h",
      "n",
      "q",
      "r"
    ],
    "filters1": 16,
    "rotation": 25,
    "dropout": 0.5,
    "budget": 24000
  },
  "6": {
    "classes": [
      "0",
      "1",
      "2",
      "3",
      "4",
      "A",
      "B",
      "C",
      "D",
      "E"
    ],
    "filters1": 24,
    "rotation": 10,
    "dropout": 0.1,
    "budget": 40000
  },
  "7": {
    "classes": [
      "5",
      "6",
      "7",
      "8",
      "9",
      "F",
      "G",
      "H",
      "I",
      "J"
    ],
    "filters1": 32,
    "rotation": 15,
    "dropout": 0.2,
    "budget": 60000
  },
  "8": {
    "classes": [
      "0",
      "1",
      "2",
      "3",
      "4",
      "K",
      "L",
      "M",
      "N",
      "O"
    ],
    "filters1": 8,
    "rotation": 20,
    "dropout": 0.3,
    "budget": 12000
  },
  "9": {
    "classes": [
      "5",
      "6",
      "7",
      "8",
      "9",
      "P",
      "Q",
      "R",
      "S",
      "T"
    ],
    "filters1": 16,
    "rotation": 25,
    "dropout": 0.4,
    "budget": 24000
  },
  "10": {
    "classes": [
      "A",
      "B",
      "C",
      "D",
      "E",
      "a",
      "b",
      "d",
      "e",
      "f"
    ],
    "filters1": 24,
    "rotation": 5,
    "dropout": 0.5,
    "budget": 40000
  },
  "11": {
    "classes": [
      "0",
      "D",
      "O",
      "Q",
      "6",
      "G",
      "8",
      "B",
      "9",
      "g"
    ],
    "filters1": 32,
    "rotation": 15,
    "dropout": 0.1,
    "budget": 60000
  },
  "12": {
    "classes": [
      "1",
      "I",
      "L",
      "7",
      "T",
      "J",
      "2",
      "Z",
      "S",
      "5"
    ],
    "filters1": 8,
    "rotation": 20,
    "dropout": 0.2,
    "budget": 12000
  },
  "13": {
    "classes": [
      "A",
      "H",
      "M",
      "N",
      "V",
      "W",
      "X",
      "Y",
      "K",
      "R"
    ],
    "filters1": 16,
    "rotation": 25,
    "dropout": 0.3,
    "budget": 24000
  },
  "14": {
    "classes": [
      "C",
      "G",
      "O",
      "Q",
      "U",
      "V",
      "D",
      "P",
      "R",
      "S"
    ],
    "filters1": 24,
    "rotation": 5,
    "dropout": 0.4,
    "budget": 40000
  },
  "15": {
    "classes": [
      "E",
      "F",
      "H",
      "I",
      "L",
      "T",
      "X",
      "Z",
      "1",
      "7"
    ],
    "filters1": 32,
    "rotation": 10,
    "dropout": 0.5,
    "budget": 60000
  },
  "16": {
    "classes": [
      "a",
      "b",
      "d",
      "e",
      "f",
      "g",
      "h",
      "n",
      "q",
      "r"
    ],
    "filters1": 8,
    "rotation": 20,
    "dropout": 0.1,
    "budget": 12000
  },
  "17": {
    "classes": [
      "A",
      "a",
      "B",
      "b",
      "D",
      "d",
      "E",
      "e",
      "F",
      "f"
    ],
    "filters1": 16,
    "rotation": 25,
    "dropout": 0.2,
    "budget": 24000
  },
  "18": {
    "classes": [
      "G",
      "g",
      "H",
      "h",
      "N",
      "n",
      "Q",
      "q",
      "R",
      "r"
    ],
    "filters1": 24,
    "rotation": 5,
    "dropout": 0.3,
    "budget": 40000
  },
  "19": {
    "classes": [
      "T",
      "t",
      "A",
      "a",
      "B",
      "b",
      "N",
      "n",
      "Q",
      "q"
    ],
    "filters1": 32,
    "rotation": 10,
    "dropout": 0.4,
    "budget": 60000
  },
  "20": {
    "classes": [
      "0",
      "2",
      "4",
      "6",
      "8",
      "A",
      "C",
      "E",
      "G",
      "I"
    ],
    "filters1": 8,
    "rotation": 15,
    "dropout": 0.5,
    "budget": 12000
  },
  "21": {
    "classes": [
      "1",
      "3",
      "5",
      "7",
      "9",
      "B",
      "D",
      "F",
      "H",
      "J"
    ],
    "filters1": 16,
    "rotation": 25,
    "dropout": 0.1,
    "budget": 24000
  },
  "22": {
    "classes": [
      "K",
      "M",
      "O",
      "Q",
      "S",
      "U",
      "W",
      "Y",
      "a",
      "d"
    ],
    "filters1": 24,
    "rotation": 5,
    "dropout": 0.2,
    "budget": 40000
  },
  "23": {
    "classes": [
      "L",
      "N",
      "P",
      "R",
      "T",
      "V",
      "X",
      "Z",
      "b",
      "e"
    ],
    "filters1": 32,
    "rotation": 10,
    "dropout": 0.3,
    "budget": 60000
  },
  "24": {
    "classes": [
      "0",
      "1",
      "A",
      "B",
      "C",
      "a",
      "b",
      "d",
      "e",
      "f"
    ],
    "filters1": 8,
    "rotation": 20,
    "dropout": 0.4,
    "budget": 12000
  },
  "25": {
    "classes": [
      "0",
      "O",
      "1",
      "L",
      "2",
      "Z",
      "5",
      "S",
      "8",
      "B"
    ],
    "filters1": 16,
    "rotation": 15,
    "dropout": 0.5,
    "budget": 24000
  }
}

# TODO: Студент — укажите свой вариант от 1 до 25.
V = 1

assert 1 <= V <= 25
CFG = VARIANTS[str(V)] if str(V) in VARIANTS else VARIANTS[V]

SELECTED_CLASSES = CFG["classes"]
FILTERS_1 = int(CFG["filters1"])
ROTATION_DEG = int(CFG["rotation"])
DROPOUT = float(CFG["dropout"])
PARAM_BUDGET = int(CFG["budget"])

SEED = 42 + V
set_seed(SEED)

print("Вариант:", V)
print("Классы:", SELECTED_CLASSES)
print("filters_1:", FILTERS_1)
print("rotation:", ROTATION_DEG)
print("dropout:", DROPOUT)
print("CNN budget:", PARAM_BUDGET)


## Шаг 3. Подготовка EMNIST

Особенности реализации:

1. используем `split="balanced"`;
2. выбираем только 10 классов варианта;
3. перенумеровываем их в `0..9`;
4. случайное вращение применяется **только к train**;
5. validation/test используют только коррекцию ориентации, `ToTensor()` и нормализацию.

In [ ]:
def fix_emnist_orientation(img):
    """Коррекция геометрии EMNIST для удобного человеческого чтения."""
    img = TF.rotate(img, -90)
    img = TF.hflip(img)
    return img

class EMNISTClassSubset(Dataset):
    """Подвыборка EMNIST с перенумерацией исходных меток в диапазон 0..C-1."""

    def __init__(self, base_dataset, indices, selected_classes, transform=None):
        self.base = base_dataset
        self.indices = list(map(int, indices))
        self.selected_classes = list(selected_classes)
        self.transform = transform

        class_to_old = {name: idx for idx, name in enumerate(base_dataset.classes)}
        self.old_to_new = {
            class_to_old[name]: new_idx
            for new_idx, name in enumerate(self.selected_classes)
        }

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        image, old_target = self.base[self.indices[i]]
        old_target = int(old_target)
        new_target = self.old_to_new[old_target]

        if self.transform is not None:
            image = self.transform(image)

        return image, new_target

def collect_indices_for_classes(base_dataset, selected_classes):
    class_to_old = {name: idx for idx, name in enumerate(base_dataset.classes)}
    selected_old = torch.tensor(
        [class_to_old[c] for c in selected_classes],
        dtype=base_dataset.targets.dtype
    )
    mask = torch.isin(base_dataset.targets, selected_old)
    return torch.where(mask)[0].cpu().numpy()

def stratified_train_val_indices(base_dataset, candidate_indices, val_fraction=0.15, seed=42):
    """Разделение по каждому классу, чтобы сохранить баланс."""
    rng = np.random.default_rng(seed)
    targets = base_dataset.targets[candidate_indices].cpu().numpy()

    train_idx, val_idx = [], []
    for cls in np.unique(targets):
        cls_indices = candidate_indices[targets == cls].copy()
        rng.shuffle(cls_indices)
        n_val = max(1, int(round(len(cls_indices) * val_fraction)))
        val_idx.extend(cls_indices[:n_val])
        train_idx.extend(cls_indices[n_val:])

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return np.array(train_idx), np.array(val_idx)

def make_transforms(rotation_deg):
    # Требуемая аугментация RandomRotation + ToTensor дополнена
    # коррекцией ориентации EMNIST и нормализацией.
    train_transform = transforms.Compose([
        transforms.Lambda(fix_emnist_orientation),
        transforms.RandomRotation(rotation_deg),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    eval_transform = transforms.Compose([
        transforms.Lambda(fix_emnist_orientation),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    return train_transform, eval_transform


In [ ]:
# Базовые EMNIST-объекты без transform:
# случайная аугментация будет применяться только через обертку train_ds.
base_train = datasets.EMNIST(
    root="./data",
    split="balanced",
    train=True,
    download=True,
    transform=None
)

base_test = datasets.EMNIST(
    root="./data",
    split="balanced",
    train=False,
    download=True,
    transform=None
)

print("Всего классов EMNIST Balanced:", len(base_train.classes))
print("Классы:", base_train.classes)

train_transform, eval_transform = make_transforms(ROTATION_DEG)

candidate_train = collect_indices_for_classes(base_train, SELECTED_CLASSES)
candidate_test = collect_indices_for_classes(base_test, SELECTED_CLASSES)

train_indices, val_indices = stratified_train_val_indices(
    base_train,
    candidate_train,
    val_fraction=0.15,
    seed=SEED
)

train_ds = EMNISTClassSubset(
    base_train, train_indices, SELECTED_CLASSES, transform=train_transform
)
val_ds = EMNISTClassSubset(
    base_train, val_indices, SELECTED_CLASSES, transform=eval_transform
)
test_ds = EMNISTClassSubset(
    base_test, candidate_test, SELECTED_CLASSES, transform=eval_transform
)

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))


In [ ]:
BATCH_SIZE = 256
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0
PIN_MEMORY = DEVICE.type == "cuda"

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
    generator=g,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
)

x_batch, y_batch = next(iter(train_loader))
print("Batch images:", x_batch.shape)
print("Batch labels:", y_batch.shape)


In [ ]:
def show_one_per_class(dataset, class_names):
    found = {}
    for image, target in dataset:
        target = int(target)
        if target not in found:
            found[target] = image
        if len(found) == len(class_names):
            break

    fig = plt.figure(figsize=(12, 3))
    for i, name in enumerate(class_names):
        ax = fig.add_subplot(2, 5, i + 1)
        img = found[i].squeeze(0).numpy()
        ax.imshow(img, cmap="gray")
        ax.set_title(name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_one_per_class(val_ds, SELECTED_CLASSES)


## Шаг 4. Реализация MLP и CNN

Заполните все участки `# TODO: Студент`. Последний слой должен возвращать **логиты**.

In [ ]:
class MLPBaseline(nn.Module):
    def __init__(self, num_classes: int, dropout: float):
        super().__init__()

        # TODO: Студент.
        # Реализуйте:
        # Flatten -> Linear(784, 64) -> ReLU -> Dropout -> Linear(64, num_classes)
        self.net = nn.Sequential(
            # nn.Flatten(),
            # ...
        )

    def forward(self, x):
        # ВАЖНО: вернуть логиты. Softmax здесь не нужен при CrossEntropyLoss.
        return self.net(x)


class EducationCNN(nn.Module):
    def __init__(self, num_classes: int, filters_1: int, dropout: float):
        super().__init__()

        # TODO: Студент.
        # Блок 1:
        # Conv2d(1, filters_1, kernel_size=3, padding=1)
        # ReLU
        # MaxPool2d(2)

        # Блок 2:
        # Conv2d(filters_1, 2*filters_1, kernel_size=3, padding=1)
        # ReLU
        # MaxPool2d(2)

        # Классификатор:
        # Flatten
        # Dropout(dropout)
        # Linear(2*filters_1*7*7, num_classes)

        self.features = nn.Sequential(
            # TODO
        )
        self.classifier = nn.Sequential(
            # TODO
        )

    def forward(self, x):
        # TODO: Студент
        raise NotImplementedError("Реализуйте forward().")


## Шаг 5. Циклы обучения и валидации

Эти функции готовы. Разберите, где выполняются прямой проход, `backward()` и шаг Adam.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, targets)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == targets).sum().item()
        total += batch_size

    return {
        "loss": running_loss / total,
        "accuracy": correct / total,
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device, return_predictions=False):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0
    y_true, y_pred = [], []

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, targets)

        preds = logits.argmax(dim=1)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        correct += (preds == targets).sum().item()
        total += batch_size

        if return_predictions:
            y_true.extend(targets.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

    result = {
        "loss": running_loss / total,
        "accuracy": correct / total,
    }

    if return_predictions:
        result["y_true"] = np.array(y_true)
        result["y_pred"] = np.array(y_pred)

    return result


def fit_model(model, train_loader, val_loader, epochs, lr, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
    }

    for epoch in range(1, epochs + 1):
        train_metrics = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_metrics = evaluate(
            model, val_loader, criterion, device
        )

        history["train_loss"].append(train_metrics["loss"])
        history["train_accuracy"].append(train_metrics["accuracy"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])

        print(
            f"Epoch {epoch:02d}: "
            f"train loss={train_metrics['loss']:.4f}, "
            f"train acc={train_metrics['accuracy']:.4f}; "
            f"val loss={val_metrics['loss']:.4f}, "
            f"val acc={val_metrics['accuracy']:.4f}"
        )

    return history


In [ ]:
# TODO: Студент — после реализации обеих моделей запустите обучение.

EPOCHS = 4
LR = 1e-3

# mlp = MLPBaseline(len(SELECTED_CLASSES), DROPOUT).to(DEVICE)
# cnn = EducationCNN(len(SELECTED_CLASSES), FILTERS_1, DROPOUT).to(DEVICE)

# print("MLP params:", count_parameters(mlp))
# print("CNN params:", count_parameters(cnn))
# assert count_parameters(cnn) <= PARAM_BUDGET

# mlp_history = fit_model(
#     mlp, train_loader, val_loader, EPOCHS, LR, DEVICE
# )

# cnn_history = fit_model(
#     cnn, train_loader, val_loader, EPOCHS, LR, DEVICE
# )


## Шаг 6. Кривые обучения

Постройте Loss и Accuracy отдельно для MLP и CNN. Объясните, есть ли признаки переобучения.

In [ ]:
def plot_loss(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, history["train_loss"], marker="o", label="Train")
    plt.plot(epochs, history["val_loss"], marker="o", label="Validation")
    plt.xlabel("Эпоха")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

def plot_accuracy(history, title):
    epochs = range(1, len(history["train_accuracy"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, history["train_accuracy"], marker="o", label="Train")
    plt.plot(epochs, history["val_accuracy"], marker="o", label="Validation")
    plt.xlabel("Эпоха")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
# TODO: Студент — после обучения постройте четыре графика:
# 1) MLP Loss
# 2) MLP Accuracy
# 3) CNN Loss
# 4) CNN Accuracy

# plot_loss(mlp_history, "MLP: функция потерь")
# plot_accuracy(mlp_history, "MLP: точность")
# plot_loss(cnn_history, "CNN: функция потерь")
# plot_accuracy(cnn_history, "CNN: точность")


## Шаг 7. Матрица ошибок

Для CNN постройте матрицу ошибок и перечислите 5 наиболее частых **направленных** путаниц `истинный → предсказанный`.

In [ ]:
def top_confusion_pairs(cm, class_names, top_k=5):
    off_diag = cm.copy()
    np.fill_diagonal(off_diag, 0)

    flat_order = np.argsort(off_diag.ravel())[::-1]
    pairs = []

    for flat_idx in flat_order:
        i, j = np.unravel_index(flat_idx, off_diag.shape)
        count = int(off_diag[i, j])
        if count <= 0:
            break
        pairs.append((class_names[i], class_names[j], count))
        if len(pairs) == top_k:
            break

    return pairs


def plot_confusion(cm, class_names, title):
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel("Предсказанный класс")
    plt.ylabel("Истинный класс")
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# TODO: Студент — получите прогнозы CNN и постройте Confusion Matrix.

# criterion = nn.CrossEntropyLoss()
# cnn_test = evaluate(
#     cnn, test_loader, criterion, DEVICE, return_predictions=True
# )
#
# cm = confusion_matrix(cnn_test["y_true"], cnn_test["y_pred"])
# plot_confusion(cm, SELECTED_CLASSES, "CNN: Confusion Matrix")
#
# print("Топ-5 направленных путаниц:")
# for true_cls, pred_cls, count in top_confusion_pairs(
#     cm, SELECTED_CLASSES, top_k=5
# ):
#     print(f"{true_cls} -> {pred_cls}: {count}")


## Шаг 8. Устойчивость к сдвигу

Сдвиньте тестовые изображения на 2 пикселя вправо и вниз и сравните падение точности MLP и CNN. Не называйте CNN строго инвариантной к сдвигу: корректный термин для свертки — **эквивариантность**, а устойчивость итоговой модели является приближенной.

In [ ]:
@torch.no_grad()
def evaluate_shifted(model, loader, device, shift_y=2, shift_x=2):
    """Оценка устойчивости к небольшому сдвигу без переобучения."""
    model.eval()

    correct = 0
    total = 0

    for images, targets in loader:
        # torch.roll циклический; зануляем появившиеся края, чтобы не было wrap-around.
        shifted = torch.roll(images, shifts=(shift_y, shift_x), dims=(2, 3))

        if shift_y > 0:
            shifted[:, :, :shift_y, :] = -1.0
        elif shift_y < 0:
            shifted[:, :, shift_y:, :] = -1.0

        if shift_x > 0:
            shifted[:, :, :, :shift_x] = -1.0
        elif shift_x < 0:
            shifted[:, :, :, shift_x:] = -1.0

        shifted = shifted.to(device)
        targets = targets.to(device)

        logits = model(shifted)
        preds = logits.argmax(dim=1)

        correct += (preds == targets).sum().item()
        total += targets.numel()

    return correct / total


In [ ]:
# TODO: Студент — сравните устойчивость к сдвигу.
# mlp_shift_acc = evaluate_shifted(mlp, test_loader, DEVICE, 2, 2)
# cnn_shift_acc = evaluate_shifted(cnn, test_loader, DEVICE, 2, 2)
#
# print("MLP shifted accuracy:", mlp_shift_acc)
# print("CNN shifted accuracy:", cnn_shift_acc)


## Шаг 9. Итоговый вывод — заполнить студенту

В отчете обязательно указать:

- итоговую accuracy MLP и CNN;
- число параметров обеих моделей;
- соблюден ли бюджет CNN;
- минимальный validation loss и максимальную validation accuracy;
- 5 наиболее частых ошибок;
- результат эксперимента со сдвигом;
- объяснение, почему CNN использует параметры эффективнее MLP;
- педагогический сценарий применения;
- ограничение: распознавание символа не должно автоматически заменять педагогическую проверку ответа учащегося.

### Таблица для отчета

| Показатель | MLP | CNN |
|---|---:|---:|
| Параметры | ... | ... |
| Test accuracy | ... | ... |
| Shifted accuracy | ... | ... |
| Падение после сдвига | ... | ... |

**Педагогический вывод:** ...

## Критерии оценивания

| Элемент | Баллы |
|---|---:|
| Датасет и подвыборка | 2 |
| Аугментация и DataLoader | 3 |
| MLP | 3 |
| CNN | 4 |
| Loss/Accuracy | 3 |
| Confusion Matrix и анализ ошибок | 3 |
| Педагогический вывод | 1 |
| Воспроизводимость | 1 |
| **Итого** | **20** |